# Nettoyage

In [11]:
import pandas as pd

In [12]:
# Chargement des données
# Utilisation de chemins relatifs au notebook 
silver_file_path = '../../shards/silver/silver_shard_1.csv'
gold_file_path = '../../shards/gold/gold_shard_1.csv'

# Lire les fichiers
df_silver = pd.read_csv(silver_file_path)
df_gold = pd.read_csv(gold_file_path)

# Ajouter une colonne 'dataset_type' pour pouvoir les séparer plus tard
df_silver['dataset_type'] = 'SILVER'
df_gold['dataset_type'] = 'GOLD'

# Concaténer les deux DataFrames
df = pd.concat([df_silver, df_gold], ignore_index=True)

# Affichage des premières lignes et des informations générales
print(f"Shape Total: {df.shape} (Silver: {df_silver.shape[0]} + Gold: {df_gold.shape[0]})")
print("Distribution des types de dataset :")
print(df['dataset_type'].value_counts())

display(df.head())
df.info()

Shape Total: (10005, 10) (Silver: 9005 + Gold: 1000)
Distribution des types de dataset :
dataset_type
SILVER    9005
GOLD      1000
Name: count, dtype: int64


,data_id,id,classe,darija_arabic,darija_arabizi,english,modern_standard_arabic,english_word_count,status,dataset_type
0,data55034,WIKI_D_06174,D,الرحيل ديال البريطانيين سالا النشاطات العسكرية...,rrahil dyal britaniyyeen sala nshatat 3askariy...,The British departure ended major military act...,أنهى الانسحاب البريطاني الأنشطة العسكرية الكبر...,32,GENERATED,SILVER
1,data55035,DATA_10281,A,كان كيسحاب ليا غادي يستعرض علينا المهارات ديالو,kan kis7ab liya ghadi yste3red 3lina lmaharat ...,i thought that he will show his skills,ظننت أنه سيظهر مهاراته,8,PARTIALLY VALIDATED,SILVER
2,data55036,WIKI_C_05466,C,الحموضة ديال المعدة هي وحدة من الإفرازات الرئي...,l7modda dyal lma3da hi wa7da mn l frazat rayis...,Gastric acid is one of the main secretions of ...,حمض المعدة هو أحد الإفرازات الرئيسية للمعدة. و...,25,GENERATED,SILVER
3,data55037,WIKI_C_00848,C,بزاف ديال الكساوي فيهم سفنج ولا شي حاجة باش يع...,bzaf dyal lksawiyy fihom sfanj wlla chi haja b...,Many suits include special padding or <unk> to...,تتضمن العديد من الأزياء حشوات خاصة أو غير ذلك ...,26,GENERATED,SILVER
4,data55038,WIKI_D_11930,D,المطاعم ديال فروم فيها الأرشينجل اللي كاين في ...,lmatam dyal from fi l'archangel lli kayen f ch...,Frome restaurants include the Archangel on Kin...,تشمل مطاعم فروم مطعم آرتشانجل في شارع كينج، وا...,42,GENERATED,SILVER


<class 'pandas.DataFrame'>
RangeIndex: 10005 entries, 0 to 10004
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   data_id                 10005 non-null  str  
 1   id                      10005 non-null  str  
 2   classe                  10005 non-null  str  
 3   darija_arabic           10005 non-null  str  
 4   darija_arabizi          10005 non-null  str  
 5   english                 10005 non-null  str  
 6   modern_standard_arabic  10005 non-null  str  
 7   english_word_count      10005 non-null  int64
 8   status                  10005 non-null  str  
 9   dataset_type            10005 non-null  str  
dtypes: int64(1), str(9)
memory usage: 8.0 MB


In [13]:
import re

# 3. Détection d'anomalies linguistiques

def has_arabic_chars(text):
    if not isinstance(text, str): return False
    # Plage Unicode pour l'arabe
    return bool(re.search(r'[\u0600-\u06FF]', text))

def has_latin_chars(text):
    if not isinstance(text, str): return False
    # Lettres latines a-z, A-Z
    return bool(re.search(r'[a-zA-Z]', text))

# Cas 1 : Anglais contenant de l'arabe
anomaly_english_with_arabic = df[df['english'].apply(has_arabic_chars)]
print(f"Lignes où la colonne 'english' contient de l'arabe : {len(anomaly_english_with_arabic)}")
if not anomaly_english_with_arabic.empty:
    display(anomaly_english_with_arabic[['id', 'english']].head())

# Cas 2 : Darija (script Arabe) contenant du Latin (Arabizi probable ou mélange)
# On peut tolérer quelques caractères latins (chiffres, ponctuation, ou noms propres), 
# mais ici on cherche une présence significative de texte latin.
anomaly_darija_with_latin = df[df['darija_arabic'].apply(has_latin_chars)]
print(f"Lignes où la colonne 'darija_arabic' contient des caractères latins : {len(anomaly_darija_with_latin)}")
if not anomaly_darija_with_latin.empty:
    display(anomaly_darija_with_latin[['id', 'darija_arabic']].head())

Lignes où la colonne 'english' contient de l'arabe : 1


,id,english
9249,wiki_2,biology\nchemistry\nmath\ntechnology\nmorocco\...


Lignes où la colonne 'darija_arabic' contient des caractères latins : 2203


,id,darija_arabic
5,WIKI_D_05628,أڤاتار آنݣ ( <unk> ، آن <unk> ) شخصية خيالية و...
7,WIKI_D_08720,المركب د التداريب د النادي كاين ف <unk> كروس، ...
9,WIKI_D_11750,واش إلى كان ؟ هو الكتاب الثاني ديال مونرو لي ت...
10,DATA_23825,1959: تفتحات لمونوريل، طوموبيلات Turnpike و وي...
11,WIKI_D_00849,ماكي، ريتشارد سومرست (1997). التجارة من وراء ا...


In [14]:
# 3.b Détection d'URLs et Emojis/Caractères Spéciaux

def has_url(text):
    if not isinstance(text, str): return False
    # Regex standard pour URL : http/https ou www.
    return bool(re.search(r'https?://\S+|www\.\S+', text))

def has_emoji_or_special(text):
    if not isinstance(text, str): return False
    # Plage Unicode approximative pour les émojis communs et symboles graphiques
    # Inclut la plage principale des emojis et quelques symboles divers
    # \U0001F600-\U0001F64F : Emoticons
    # \U0001F300-\U0001F5FF : Misc Symbols and Pictographs
    # \U0001F680-\U0001F6FF : Transport and Map
    # \u2700-\u27BF : Dingbats
    return bool(re.search(r'[\U0001F300-\U0001FAFF]|[\u2700-\u27BF]', text))

# Detection URLs
anomaly_urls = pd.DataFrame()
for col in ['darija_arabic', 'darija_arabizi', 'english']:
    mask = df[col].apply(has_url)
    if mask.any():
        print(f"URLs détectées dans la colonne '{col}' : {mask.sum()}")
        subset = df[mask].copy()
        subset['anomaly_type'] = f'URL in {col}'
        anomaly_urls = pd.concat([anomaly_urls, subset])

if not anomaly_urls.empty:
    print("\nExemple de lignes contenant des URLs :")
    display(anomaly_urls[['id', 'english', 'darija_arabic', 'anomaly_type']].head())
else:
    print("Aucune URL détectée.")


# Detection Emojis
anomaly_emojis = pd.DataFrame()
for col in ['darija_arabic', 'darija_arabizi', 'english']:
    mask = df[col].apply(has_emoji_or_special)
    if mask.any():
        print(f"Emojis/Symboles détectés dans la colonne '{col}' : {mask.sum()}")
        subset = df[mask].copy()
        subset['anomaly_type'] = f'Emoji in {col}'
        anomaly_emojis = pd.concat([anomaly_emojis, subset])

if not anomaly_emojis.empty:
    print("\nExemple de lignes contenant des Emojis/Symboles :")
    display(anomaly_emojis[['id', 'english', 'darija_arabic', 'anomaly_type']].head())
else:
    print("Aucun emoji détecté.")

URLs détectées dans la colonne 'darija_arabic' : 1
URLs détectées dans la colonne 'darija_arabizi' : 4
URLs détectées dans la colonne 'english' : 1

Exemple de lignes contenant des URLs :


,id,english,darija_arabic,anomaly_type
2911,WIKI_C_04365,with Bertrand Russell . Principia Mathematica ...,مع برتراند راسل. برينسيپيا ماثيماتيكا، الجزء ا...,URL in darija_arabic
2911,WIKI_C_04365,with Bertrand Russell . Principia Mathematica ...,مع برتراند راسل. برينسيپيا ماثيماتيكا، الجزء ا...,URL in darija_arabizi
3475,WIKI_C_04368,The Concept of Nature . Cambridge : Cambridge ...,مفهوم الطبيعة. كامبريدج: مطبعة جامعة كامبريدج،...,URL in darija_arabizi
7401,WIKI_C_04369,The Principle of Relativity with Applications ...,لمبدا ديال النسبية والتطبيقات ديالو ف العلوم ا...,URL in darija_arabizi
8382,WIKI_C_01426,"As a result of his contributions , Francois Va...","بسباب الإسهامات ديالو، فرانكوا فالي تسمى ""المد...",URL in darija_arabizi


Aucun emoji détecté.


In [15]:
# Nettoyage approfondi et Normalisation
# Suppression : URLs, Emojis, <unk>, et excès d'espaces
import re

string_cols = ['darija_arabic', 'darija_arabizi', 'english', 'modern_standard_arabic']
url_pattern = r'https?://\S+|www\.\S+'
# CORRECTION : Ne pas utiliser 'r' (raw string) ici. 
# On veut que Python convertisse \U... en vrais caractères avant de passer le regex.
emoji_pattern = '[\U0001F300-\U0001FAFF]|[\u2700-\u27BF]' 

unk_pattern = r'<unk>' # Token spécifique à nettoyer
# Artifacts de tokenization fréquents (ex: @-@, @,@) vus dans le fichier silver
artifact_pattern = r'\s@[-|,|.]@\s' 

def normalize_punctuation(text):
    if not isinstance(text, str): return text
    # Unifier les guillemets et apostrophes (curly quotes -> straight quotes)
    text = re.sub(r'[\u2018\u2019]', "'", text)
    text = re.sub(r'[\u201C\u201D]', '"', text)
    
    # Supprimer ponctuation répétée (ex: "!!" -> "!", "..." -> ".")
    # On garde 1 seule occurrence pour .,!?،؟
    text = re.sub(r'([.,!?،؟])\1+', r'\1', text)
    return text

print("Démarrage du nettoyage approfondi...")

for col in string_cols:
    if col in df.columns:
        # Gestion des NaN : Remplacer par vide pour éviter la chaine "nan"
        series = df[col].fillna('').astype(str)
        
        # 1. Suppression des URLs
        series = series.str.replace(url_pattern, ' ', regex=True)
        
        # 2. Suppression des Emojis
        # Note: Si le backend regex de pandas (pyarrow) pose problème avec les ranges unicode complexes, 
        # une alternative est d'utiliser `apply` avec `re.sub` de Python standard.
        try:
             series = series.str.replace(emoji_pattern, ' ', regex=True)
        except Exception as e:
            print(f"Fallback vers apply(re.sub) pour la colonne {col} à cause de : {e}")
            series = series.apply(lambda x: re.sub(emoji_pattern, ' ', x))
        
        # 3. Suppression de <unk> (case insensitive via case=False)
        series = series.str.replace(unk_pattern, ' ', regex=True, case=False)

        # 4. Suppression des artifacts de tokenization (ex: " @-@ ")
        series = series.str.replace(artifact_pattern, '-', regex=True)
        
        # 5. Normalisation de la ponctuation (NOUVEAU)
        series = series.apply(normalize_punctuation)
        
        # 6. Suppression des espaces superflus (trim + double espaces)
        series = series.str.strip().str.replace(r'\s+', ' ', regex=True)
        
        # Remettre les chaines vides à NaN si nécessaire, ou garder vide. 
        # Ici on garde les chaines nettoyées.
        df[col] = series

print("Nettoyage terminé.")

# Vérification après nettoyage
print("Aperçu après nettoyage (<unk> et artefacts retirés) :")
# Afficher quelques lignes qui contenaient potentiellement des anomalies
if not anomaly_darija_with_latin.empty:
     display(df.loc[anomaly_darija_with_latin.index, string_cols].head())

Démarrage du nettoyage approfondi...
Nettoyage terminé.
Aperçu après nettoyage (<unk> et artefacts retirés) :


,darija_arabic,darija_arabizi,english,modern_standard_arabic
5,أڤاتار آنݣ ( ، آن ) شخصية خيالية وهو الشخصية ا...,"avatar aang ( , aan ) shakhsiya khayaliya w ho...","Avatar Aang ( , ān ) is a fictional character ...",أفاتار آنغ ( ، آن ) شخصية خيالية والبطل الرئيس...
7,المركب د التداريب د النادي كاين ف كروس، ف طريق...,"l merkeb d tdarib d n-nadi kayen f kros, f tar...","The club 's training ground is Cross , in Gran...",يقع ملعب تدريب النادي في كروس، في طريق غرانج، ...
9,واش إلى كان ؟ هو الكتاب الثاني ديال مونرو لي ت...,wash ila kan? huwa lktab t-tani dyal munro li ...,"What If ? is Munroe 's second published book ,...",ماذا لو؟ هو الكتاب الثاني المنشور لـ مونرو، وك...
10,1959: تفتحات لمونوريل، طوموبيلات Turnpike و وي...,"1959: tft7at lmonorail, tomobilaat Turnpike o ...","1959 : Monorail , Turnpike Cars , and Wild Mou...",١٩٥٩: يفتح قطار المونوريل، وسيارات التورنبايك،...
11,ماكي، ريتشارد سومرست (1997). التجارة من وراء ا...,"makki, richard somerset (1997). ttijara mn war...","Mackie , Richard Somerset ( 1997 ) . Trading B...",ماكي، ريتشارد سومرست (1997). التجارة من وراء ا...


In [16]:
# Séparation des datasets après nettoyage

print("Séparation des datasets...")
df_silver_clean = df[df['dataset_type'] == 'SILVER'].copy()
df_gold_clean = df[df['dataset_type'] == 'GOLD'].copy()

# Supprimer la colonne temporaire 'dataset_type'
df_silver_clean.drop(columns=['dataset_type'], inplace=True)
df_gold_clean.drop(columns=['dataset_type'], inplace=True)

print(f"Shape Silver après nettoyage: {df_silver_clean.shape}")
print(f"Shape Gold après nettoyage: {df_gold_clean.shape}")

# Sauvegarde des fichiers nettoyés (Optionnel)
df_silver_clean.to_csv('silver_shard_1_clean.csv', index=False)
df_gold_clean.to_csv('gold_shard_1_clean.csv', index=False)
print("Fichiers nettoyés sauvegardés.")

Séparation des datasets...
Shape Silver après nettoyage: (9005, 9)
Shape Gold après nettoyage: (1000, 9)
Fichiers nettoyés sauvegardés.
